# Lesson 05 - Connect to an existing Foundry agent

**Level:** intermediate | **Demo time:** 8 minutes | **Builds on:** Lessons 01 and 04

`FoundryAgent` connects the Agent Framework programming model to an agent definition managed by Microsoft Foundry Agent Service. The agent's model, instructions, hosted tools, and version live in Foundry rather than in this notebook.

## Learning objectives

- Connect to a pre-created Prompt Agent or Hosted Agent by name.
- Optionally pin a Prompt Agent version.
- Reuse MAF run, streaming, and session APIs with a service-managed definition.

## Choose the ownership model intentionally

| Application-owned | Foundry-managed |
|---|---|
| `Agent(client=FoundryChatClient(...))` | `FoundryAgent(...)` |
| Instructions and tools live in Python | Definition and hosted tools live in Foundry |
| Changes deploy with the application | Prompt Agent definitions can be centrally versioned |
| Best for dynamic run-time composition | Best for governed, reusable agent definitions |

A Hosted Agent needs only its name. A versioned Prompt Agent can also use `FOUNDRY_AGENT_VERSION`; leaving it blank selects the latest applicable definition.

## Before running

Create or select an agent in the same Foundry project, then set:

```dotenv
FOUNDRY_AGENT_NAME=your-agent-name
FOUNDRY_AGENT_VERSION=1  # optional; leave blank for a Hosted Agent
```

This lesson never creates, updates, or deletes a Foundry agent.

In [ ]:
from agent_framework.foundry import FoundryAgent

from maf_foundry_hackathon.config import WorkshopSettings, create_credential

settings = WorkshopSettings.from_env(require_agent=True)
credential = create_credential()
settings.safe_summary()
print(f"Settings loaded successfully with agent: {settings.agent_name} version {settings.agent_version}")

In [ ]:
agent_options = {
    "project_endpoint": settings.project_endpoint,
    "agent_name": settings.agent_name,
    "credential": credential,
}
if settings.agent_version:
    agent_options["agent_version"] = settings.agent_version

foundry_agent = FoundryAgent(**agent_options)
print(f"Connected to Foundry agent: {settings.agent_name}")

Notice that the constructor does not replace the stored model or instructions. The Foundry definition is authoritative. Local Python callables are passed only when a Prompt Agent declares a matching local function tool that this client must execute.

In [ ]:
question = "Tell me a little bit about Project Aurora"
response = await foundry_agent.run(question)

print("User:", question)
print("Agent:", response.text)

In [ ]:
session = foundry_agent.create_session()

first = await foundry_agent.run(
    "What are project Aurora's business goals?",
    session=session,
)
second = await foundry_agent.run(
    "Can you highlight some of its risks?",
    session=session,
)

print("First response:", first.text)
print("Follow-up:", second.text)

## Operational guidance

- Pin a Prompt Agent version when reproducibility matters.
- Test a new version before moving callers to it.
- Keep agent names, versions, and session identifiers in trusted server-side state.
- Use application-owned agents when tools or instructions must change dynamically per request.
- Do not let an end user choose an arbitrary agent name or conversation ID.

In [ ]:
credential.close()

## Recap

MAF offers one programming model across local definitions and Foundry-managed definitions. Lesson 06 composes multiple specialized application-owned agents into a handoff workflow.